# 3.1 章节介绍：FA 算子代码结构

欢迎来到 **FlashAttention 算子开发课程** 的第 3 章！

第 2 章里，我们已经把 FlashAttention 的数学原理彻底吃透：Online Softmax 的三步递推、分块双层循环、FA1 到 FA2 的手术式改进。现在摆在面前的是一座真实的代码大厦——开源算子仓库 [ops-transformer](https://gitcode.com/cann/ops-transformer/tree/master/attention/fused_infer_attention_score) 中的 `FusedInferAttentionScore`（下文简称 FA 算子）。

> FA 算子是典型的**融合算子**：一次调用完成「矩阵乘 → Softmax → 掩码 → 矩阵乘 → 状态更新」全流程。它的代码量大、模块繁多，直接扎进源码很容易迷路。

但从一个**调用者的视角**看，再复杂的算子也只有三层：

| 层次 | 职责 | 类比 |
|--|--|--|
| **算子调用通路** | 从框架 / aclnn API 进来，做合法性检查、准备输入，最终发起算子执行 | 大楼的「接待处」 |
| **Tiling** | 在 Host（CPU）侧算出所有切分参数，打包成 TilingData 下发 | 大楼的「施工图纸」 |
| **Kernel** | 在 Device（NPU）侧按图施工：多核切分、四拍循环、流水并行 | 大楼的「施工现场」 |

本章以**基础极简 FA 代码**为例，自顶向下拆解这三层，给你一份「读源码的地图」。

## 本章学习目标

学完第 3 章，你将能够：

1. **画出算子调用通路**：说清从框架 / `aclnn` API 到算子执行的完整链路，以及 FA 算子十几个核心参数各自的作用。
2. **辨别四种数据排布**：BSND / BNSD / BSH / TND 各自的存储顺序、内存特点与适用场景；GQA 场景下排布如何变化。
3. **理解 Tiling 的存在意义**：为什么切分参数要在 Host 侧算好？Scalar 资源、ICache 容量、图开销分别如何约束 Tiling 设计？
4. **走读 Tiling 源码**：按「取输入 → 取平台 → 判支持 → 选基本块 → 核内外切分 → 算子库 tiling → workspace → 后处理 → TilingKey」八步读懂一个真实模板。
5. **看懂 Kernel 骨架**：核间切块按哪些轴切、负载均衡如何分核、核内如何沿 S2 轴切块驱动四拍循环（mm1 → vector1 → mm2 → vector2）。
6. **理解流水设计**：乒乓内存如何让「搬运」与「计算」重叠？Preload 如何提前启动 mm1 / vector1？

## 前置知识

| 知识点 | 要求 | 对应章节 |
|--|--|--|
| FA 数学原理 | Online Softmax、分块循环、FA1/FA2 差异 | [第 2 章](../02_fa_principle/README.md) |
| 昇腾 AI Core 架构 | Cube/Vector 分离架构、存储层次（GM/L2/L1/L0/UB） | [第 1 章](../01_ascend_dev_basics/README.md) |
| Ascend C 编程模型 | SPMD 多核并行、TQue 流水线范式 | [第 1 章](../01_ascend_dev_basics/README.md) |
| C++ | 能读懂类、继承、模板 | 不需要动手写 |

> 本章全部是**代码阅读课**，无需 NPU 环境；建议对照 ops-transformer 仓库源码同步阅读。

## 章节内容导航

本章共 8 个 Notebook，建议按顺序学习：

| Notebook | 主题 | 核心内容 |
|--|--|--|
| [3.1 章节介绍](03.01_chapter_intro.ipynb)（本页） | 导学 | 目标、前置、路线图 |
| [3.2 算子调用通路](03.02_call_path.ipynb) | 接待处 | aclnn 两段式接口、FA 算子参数表、直调流程 |
| [3.3 数据排布格式](03.03_data_layout.ipynb) | 数据形状 | BSND / BNSD / BSH / TND、尾块与 padding、GQA 排布 |
| [3.4 Tiling 设计原则](03.04_tiling_design.ipynb) | 图纸原则 | 五大设计原则、模板优先级调度、Cube/Vector 切分权衡 |
| [3.5 Tiling 源码走读](03.05_tiling_code_walkthrough.ipynb) | 图纸细节 | 单模板八步流程逐段读代码 |
| [3.6 Kernel 整体框架](03.06_kernel_structure.ipynb) | 施工总览 | Kernel 入参、SPMD、核间切块与负载均衡 |
| [3.7 核内切块与流水](03.07_kernel_pipeline.ipynb) | 施工细节 | 四拍循环、MM1/MM2 搬运链路、乒乓与 Preload |
| [3.8 章节测验](03.08_chapter_test.ipynb) | 检验 | 综合练习 + 答案索引 |

### 章节主线：一次算子调用的一生

```text
3.2 框架调用 aclnn FA 接口
        │  检查输入、申请 workspace、构造 executor
        ▼
3.3 输入张量按 BSND/BNSD/BSH/TND 之一排布，
   决定了「取一个基本块」是顺序取还是跳着取
        ▼
3.4 Host 侧 TilingFunc 被触发：
   按优先级逐个尝试模板，选出基本块与切分方案
        ▼
3.5 命中的模板八步走完，产出 TilingData + TilingKey
        ▼
3.6 Kernel 在 NPU 上启动：blockDim 个组合核，
   按 B/N/S1 轴核间切块，负载均衡分派
        ▼
3.7 每个核沿 S2 轴核内切块，四拍循环 + 乒乓 + preload
        ▼
3.8 综合测验：合上书，把这张地图默画一遍
```

## 学习建议

- **三层视角随时切换**：读源码时先问自己「这行代码属于调用通路、Tiling 还是 Kernel？」定位层次比读懂细节更重要。
- **代码不必逐行抠**：本章目标是「读懂骨架 + 记住关键设计动机」，第 5 章实战时再回头精读。

准备好之后，进入 [3.2 算子调用通路](03.02_call_path.ipynb)，看看一次 FA 调用从进门开始都发生了什么。